In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from torchvision import models
from torchvision import transforms


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [3]:
X = np.load('../data/X.npy', mmap_mode='r')
y = np.load('../data/y.npy', mmap_mode='r')
print(X.shape, y.shape)

(28709, 224, 224, 3) (28709,)


In [4]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
])

In [5]:
class EmotionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.from_numpy(np.array(self.images[idx], copy=True)).permute(2, 0, 1).float() / 255.0
        if self.transform is not None:
            image = self.transform(image)
        image = (image - mean) / std
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

In [6]:
indices = np.arange(len(y))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=np.asarray(y)
)

In [7]:
train_loader = DataLoader(
    Subset(EmotionDataset(X, y, transform=train_transform), train_indices),
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    Subset(EmotionDataset(X, y), val_indices),
    batch_size=64,
    num_workers=0,
    pin_memory=True
)

print(f'Train batches: {len(train_loader)}, validation batches: {len(val_loader)}')

Train batches: 359, validation batches: 90


In [8]:
unique, counts = np.unique(y[train_indices], return_counts=True)
class_weights = 1.0 / torch.tensor(counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * len(unique)
class_weights = class_weights.to(device)

print("Class weights:", class_weights)

Class weights: tensor([0.4802, 4.3972, 0.4683, 0.2659, 0.3864, 0.3972, 0.6049])


In [9]:
model = models.resnet34(pretrained=True)

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 7)
)

model = model.to(device)

C:\Users\kusha\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\kusha\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
for param in model.parameters():
    param.requires_grad = False

for param in model.layer3.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

In [11]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3,
)

In [12]:
best_val_acc = 0
patience = 7
patience_counter = 0

In [13]:
for epoch in range(20):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (batch_X, batch_y) in enumerate(train_loader):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_X.size(0)

        _, predicted = torch.max(preds, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

        if i % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_acc = correct / total

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            preds = model(batch_X)
            _, predicted = torch.max(preds, 1)

            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    val_acc = correct / total
    avg_loss = total_loss / len(train_loader.dataset)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.3f}, Train Acc: {train_acc:.3f}, Val Acc: {val_acc:.3f}")

    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'emotion_model_v4.pth')
        print(f"New best model saved! Val Acc: {val_acc:.3f}")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

C:\Users\kusha\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1 | Batch 0/359 | Loss: 2.5524
Epoch 1 | Batch 50/359 | Loss: 2.2960
Epoch 1 | Batch 100/359 | Loss: 2.3128
Epoch 1 | Batch 150/359 | Loss: 2.2305
Epoch 1 | Batch 200/359 | Loss: 2.2491
Epoch 1 | Batch 250/359 | Loss: 2.2245
Epoch 1 | Batch 300/359 | Loss: 1.9763
Epoch 1 | Batch 350/359 | Loss: 2.1959
Epoch 1, Loss: 2.136, Train Acc: 0.121, Val Acc: 0.098
New best model saved! Val Acc: 0.098
Epoch 2 | Batch 0/359 | Loss: 2.1299
Epoch 2 | Batch 50/359 | Loss: 2.0312
Epoch 2 | Batch 100/359 | Loss: 1.8699
Epoch 2 | Batch 150/359 | Loss: 1.9387
Epoch 2 | Batch 200/359 | Loss: 2.0729
Epoch 2 | Batch 250/359 | Loss: 1.7853
Epoch 2 | Batch 300/359 | Loss: 1.9974
Epoch 2 | Batch 350/359 | Loss: 2.0531
Epoch 2, Loss: 1.999, Train Acc: 0.225, Val Acc: 0.346
New best model saved! Val Acc: 0.346
Epoch 3 | Batch 0/359 | Loss: 1.8907
Epoch 3 | Batch 50/359 | Loss: 1.8242
Epoch 3 | Batch 100/359 | Loss: 2.0318
Epoch 3 | Batch 150/359 | Loss: 1.8959
Epoch 3 | Batch 200/359 | Loss: 1.9647
Epoch 

In [14]:
print(f"\nBest validation accuracy: {best_val_acc:.3f}")


Best validation accuracy: 0.588
